# TEMA + MACD Ensemble — BTC-USD

Fixed-parameter **ensemble** of:

1. **TEMA** (triple-exponential MA) — trend filter: price above rising TEMA → bullish
2. **MACD** (12/26/9) — momentum: histogram & line vs signal

**Ensemble modes**

- `weighted` (default): 50/50 score → long if score ≥ 0.35, short if ≤ −0.35
- `consensus`: long only when **both** TEMA and MACD are bullish (reduces whipsaw)

Unlike `TEMA-TEMPLATE_adjusted6_warmup.ipynb`, this notebook **does not grid-search** parameters on BTC (classic overfitting path). Defaults are fixed; a small **stability sweep** checks neighbors only.

Execution lag: `position = signal.shift(1)` — trade next bar.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from strategies.tema_macd_ensemble_btc import (
    EnsembleConfig,
    build_ensemble_frame,
    load_btc_close,
    summarize_backtest,
    tema,
    macd,
)

plt.style.use("dark_background")


In [ ]:
CFG = EnsembleConfig(
    ticker="BTC-USD",
    tema_period=55,
    macd_fast=12,
    macd_slow=26,
    macd_signal=9,
    tema_weight=0.5,
    macd_weight=0.5,
    mode="weighted",
    long_only=True,  # BTC store-of-value; set False for long/short
    cost_bps=10.0,
    ann_days=365,
)

close = load_btc_close("2018-01-01")
print(f"Bars: {len(close)} | {close.index[0].date()} → {close.index[-1].date()}")


In [ ]:
WARMUP = max(CFG.tema_period * 3, CFG.macd_slow + CFG.macd_signal) + 5
TRAIN_RATIO = 0.60
split = int(len(close) * TRAIN_RATIO)

train = close.iloc[:split]
test = close.iloc[split:]

train_df = build_ensemble_frame(train, CFG).iloc[WARMUP:]
test_seed = close.iloc[split - WARMUP :]
test_df = build_ensemble_frame(test_seed, CFG).iloc[WARMUP:]

train_stats = summarize_backtest(train_df, CFG.ann_days)
test_stats = summarize_backtest(test_df, CFG.ann_days)

print("In-sample:", train_stats)
print("Out-of-sample:", test_stats)


In [ ]:
full_df = build_ensemble_frame(close, CFG).iloc[WARMUP:]
full_stats = summarize_backtest(full_df, CFG.ann_days)
print("Full sample:", full_stats)

last = full_df.iloc[-1]
print("\nLatest bar signal:")
print(f"  Close: {last['close']:.2f}")
print(f"  TEMA({CFG.tema_period}): {last['tema']:.2f} | sig={last['sig_tema']:+.0f}")
print(f"  MACD hist: {last['macd_hist']:.2f} | sig={last['sig_macd']:+.0f}")
print(f"  Ensemble score: {last['ensemble_score']:.2f}")
print(f"  Position (next bar): {last['position']:+.0f}")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

ax = axes[0]
ax.plot(full_df.index, full_df["close"], label="BTC-USD", color="white", lw=1.2)
ax.plot(full_df.index, full_df["tema"], label=f"TEMA({CFG.tema_period})", color="#59c2ff", lw=1)
ax.set_title("BTC — TEMA trend filter")
ax.legend(loc="upper left")

ax = axes[1]
ax.plot(full_df.index, full_df["macd"], label="MACD", color="#ffb454")
ax.plot(full_df.index, full_df["macd_signal"], label="Signal", color="#c792ea")
ax.bar(full_df.index, full_df["macd_hist"], label="Hist", alpha=0.35, color="#3dd68c")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("MACD 12/26/9")
ax.legend(loc="upper left")

ax = axes[2]
ax.plot(full_df.index, full_df["equity"], label="Ensemble equity", color="#3dd68c", lw=1.5)
bh = (1 + full_df["close"].pct_change().fillna(0)).cumprod()
ax.plot(full_df.index, bh, label="Buy & hold", color="gray", alpha=0.7, lw=1)
ax.set_title(f"Ensemble ({CFG.mode}) — Sharpe {full_stats.get('sharpe', float('nan')):.2f}")
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
# Consensus mode comparison (both TEMA + MACD must agree)
cfg_consensus = EnsembleConfig(mode="consensus")
consensus_df = build_ensemble_frame(close, cfg_consensus).iloc[WARMUP:]
consensus_stats = summarize_backtest(consensus_df, CFG.ann_days)

pd.DataFrame(
    [
        {"mode": "weighted", **full_stats},
        {"mode": "consensus", **consensus_stats},
    ]
).set_index("mode")


In [ ]:
# Stability sweep: TEMA period neighbors only (not a full grid search)
rows = []
base = CFG.tema_period
for p in range(base - 10, base + 11, 5):
    if p < 10:
        continue
    c = EnsembleConfig(tema_period=p)
    df_p = build_ensemble_frame(close, c).iloc[WARMUP:]
    st = summarize_backtest(df_p, c.ann_days)
    rows.append({"tema_period": p, **st})

pd.DataFrame(rows).sort_values("tema_period")
